Extracting API levels

In [ ]:
import os
import yaml
import pandas as pd
import re

# === CONFIG ===
PROJECTS_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\July_29\YAML_Files"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_29"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_CSV = os.path.join(OUTPUT_DIR, "3.3_Project_List_API_ShallowC.csv")
DETAILED_CSV = os.path.join(OUTPUT_DIR, "3.3_Project_List_API_Details_ShallowC.csv")


# === EXTRACT ALL API LEVELS (unchanged) ===
def extract_all_api_levels(obj):
    api_levels = set()

    def recurse(o):
        if isinstance(o, dict):
            for k, v in o.items():
                key_lower = str(k).lower()
                if key_lower in ['api-level', 'api'] or key_lower.startswith('android-'):
                    if isinstance(v, list):
                        for val in v:
                            if str(val).isdigit():
                                api_levels.add(str(val))
                    elif isinstance(v, (int, str)) and str(v).isdigit():
                        api_levels.add(str(v))
                else:
                    recurse(v)
        elif isinstance(o, list):
            for item in o:
                recurse(item)
        elif isinstance(o, str):
            matches = re.findall(r'\b(?:api(?:-level)?|targetSdk|compileSdk)[\s:=]*["\']?(\d{2,3})["\']?', o, flags=re.IGNORECASE)
            for match in matches:
                api_levels.add(match)

    recurse(obj)
    return api_levels


# === PARSE YAML FILE (unchanged) ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            content = yaml.safe_load(raw)
            if not content:
                return {'api_levels': set(), 'error': True}
            all_api_levels = extract_all_api_levels(content)
            return {'api_levels': all_api_levels, 'error': False}
    except Exception:
        return {'api_levels': set(), 'error': True}


# === SCAN PROJECTS ===
project_results = {}
detailed_rows = {}

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)

            # ✅ Extract full_name before first "__"
            full_name = filename.split("__")[0].lower()
            result = parse_yaml_file(file_path)

            if full_name not in project_results:
                project_results[full_name] = {
                    'api_levels': set(),
                    'errors': 0,
                    'yml_count': 0
                }
                detailed_rows[full_name] = []

            project_results[full_name]['api_levels'].update(result['api_levels'])
            project_results[full_name]['yml_count'] += 1
            if result['error']:
                project_results[full_name]['errors'] += 1

# === BUILD DETAILED ROWS ===
final_detailed_rows = []
for full_name, data in project_results.items():
    for api in data['api_levels']:
        final_detailed_rows.append({
            'filename': '',  # placeholder for below loop
            'full_name': full_name,
            'api_level': api,
            'source': 'detected',
            'yml_count': data['yml_count']
        })

# === Rebuild detailed rows with filename ===
# Note: Since each full_name may come from multiple files,
# we'll loop again to map filename properly.

detailed_output = []
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            filename = os.path.basename(file)
            full_name = filename.split("__")[0].lower()

            result = parse_yaml_file(os.path.join(root, file))
            for api in result['api_levels']:
                detailed_output.append({
                    'filename': filename,
                    'full_name': full_name,
                    'api_level': api,
                    'source': 'detected',
                })


# === EXPORT ===
df_detailed = pd.DataFrame(detailed_output)
df_detailed.to_csv(DETAILED_CSV, index=False)

summary_rows = []
for full_name, result in project_results.items():
    summary_rows.append({
        'full_name': full_name,
        'distinct_api_levels': len(result['api_levels']),
        'yaml_errors': result['errors']
    })

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

print(f"\n✅ Summary CSV saved to: {SUMMARY_CSV}")
print(f"✅ Detailed CSV saved to: {DETAILED_CSV}")



✅ Summary CSV saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type 1\3.3_Project_List_API_ShallowC.csv
✅ Detailed CSV saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type 1\3.3_Project_List_API_Details_ShallowC.csv
